## 下載前處理，把需要的欄位：「firm_ID_Annual、CIK、cik_char、GVKEY、HTTP_NAME_HTML、HTTP_NAME_TEXT」留下

In [1]:
import pandas as pd
import time

start = time.perf_counter()
KEEP = ["coname","cik", "accession"]


def keep_and_dedupe(df):
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    return out[KEEP].drop_duplicates(subset=["cik", "accession"], keep="first")


df = pd.read_excel("/Users/wanghao/8k/8K_Filings.xlsx", sheet_name=0)
res = keep_and_dedupe(df)
print(f"刪減後（列, 欄）= {res.shape}")
end = time.perf_counter()
print(f"花費時間：{end - start:.2f} 秒")

刪減後（列, 欄）= (835233, 3)
花費時間：40.03 秒


## 查看是有否缺失值

In [3]:
res.isna().sum()

coname       0
cik          0
accession    0
dtype: int64

## 爬蟲並下載

In [1]:
#可用的版本
import os
import time
from typing import Iterable, List, Optional
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

SEC_PREFIX = "https://www.sec.gov/"
USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome Safari (Contact: haowang5277@gmail.com)"
)
OUTPUT_DIR = "/Users/wanghao/Downloads/test"

class FatalError(RuntimeError):
    pass

def _full_url(href: str) -> str:
    if not href or str(href).strip().lower() in {"", "nan", "none"}:
        raise FatalError("空的 href，無法組完整 URL")
    return urljoin(SEC_PREFIX, href)


MIN_INTERVAL = 0.12  # 秒
_last_call_ts = 0.0

def _rate_limit():
    """控制請求速率，避免超過 SEC 門檻"""
    global _last_call_ts
    now = time.monotonic()
    wait = (_last_call_ts + MIN_INTERVAL) - now
    if wait > 0:
        time.sleep(wait)
    _last_call_ts = time.monotonic()

def _get(url: str) -> requests.Response:
    """GET 請求，支援速率限制與 Retry-After 顯示"""
    _rate_limit()  # 每次請求前限速
    try:
        r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30)

        # 處理 429 Too Many Requests
        if r.status_code == 429:
            from pprint import pprint
            print(f"[429] Too Many Requests: {url}")
            pprint(dict(r.headers))  # 印出所有 Header
            ra = r.headers.get("Retry-After")
            try:
                sleep_s = int(ra) if ra and ra.isdigit() else 5
            except ValueError:
                sleep_s = 5
            print(f"等待 {sleep_s} 秒後重試...")
            time.sleep(sleep_s)
            _rate_limit()
            r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30)

        if r.status_code != 200:
            raise FatalError(f"HTTP {r.status_code}: {url}")
        return r

    except requests.RequestException as e:
        raise FatalError(f"請求失敗: {url}: {e}")


from urllib.parse import urlparse, parse_qs

def _find_10k_link(index_html: str) -> str:
    soup = BeautifulSoup(index_html, "html.parser")

    # 找到 Document Format Files 表格
    table = None
    for t in soup.find_all("table"):
        ths = [th.get_text(strip=True).lower() for th in t.find_all("th")]
        if {"seq", "description", "document", "type", "size"} <= set(ths):
            table = t
            break
    if table is None:
        raise FatalError("找不到 Document Format Files 表格")

    # 欄位索引
    ths = [th.get_text(strip=True).lower() for th in table.find_all("th")]
    col = {name: ths.index(name) for name in ["seq","description","document","type","size"]}

    keywords = ("8-k","10-k","20-f","40-f","n-csr","1-k","ar","6-k","10ksb")

    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) <= max(col.values()):
            continue

        type_txt = tds[col["type"]].get_text(strip=True).lower()
        if not any(k in type_txt for k in keywords):
            continue

       # /ix?doc= 解包
        for a in tds[col["document"]].find_all("a", href=True):
            raw = a["href"].strip()
            low = raw.lower()

            # 1) 若為 /ix?doc=...，把 doc 參數取出來（真正檔案）
            if low.startswith("/ix?") or "/ix?" in low:
                q = urlparse(raw).query
                doc = parse_qs(q).get("doc", [None])[0]
                if doc:
                    raw = doc
                    low = raw.lower()

            # 2) 仍要避開純 viewer 殼檔（沒有 doc 參數、也不是實體檔案）
            if "viewer" in low and ("doc=" not in low) and not low.endswith((".htm", ".html", ".txt")):
                continue

            # 3) 是實體檔就回傳
            if low.endswith((".htm", ".html", ".txt")):
                return urljoin(SEC_PREFIX, raw)

    raise FatalError("找不到符合 keywords 的主檔連結")




def download_10k_from_df(df: pd.DataFrame, out_dir: str = OUTPUT_DIR, limit: int = 100) -> List[str]:
    urls = df["HTTP_NAME_HTML"].astype(str).tolist()[:limit]
    firm_ids = df["firm_ID_Annual"].astype(str).tolist()[:limit]
    return download_10k_from_urls(urls=urls, out_dir=out_dir, firm_ids=firm_ids)

def download_10k_from_urls(
    urls: Iterable[str],
    out_dir: str = OUTPUT_DIR,
    firm_ids: Optional[Iterable[str]] = None
) -> List[str]:
    # 直接使用既有的三個資料夾
    html_dir = "/Volumes/One Touch/10_k_file_HTML"
    txt_dir = "/Volumes/One Touch/10_k_file_TXT"
    other_dir = "/Volumes/One Touch/10_k_file_Neither"

    start_t = time.time()
    urls = list(urls)

    if firm_ids is None:
        ids = [f"firm_{i:06d}" for i in range(len(urls))]
    else:
        ids = list(firm_ids)
        if len(ids) != len(urls):
            print(f"[警告] firm_ids({len(ids)}) 與 urls({len(urls)}) 數量不一致，將自動補齊/截斷")
            ids = (ids + [f"firm_{i:06d}" for i in range(len(ids), len(urls))])[:len(urls)]

    saved: List[str] = []
    errors: List[dict] = []

    for i, raw in enumerate(urls):
        firm_id = ids[i]
        try:
            index_url = _full_url(raw)
            index_resp = _get(index_url)
            tenk_url = _find_10k_link(index_resp.text)
            tenk_resp = _get(tenk_url)

            # 判斷副檔名存到不同資料夾
            if tenk_url.lower().endswith((".htm", ".html")):
                file_name = f"{firm_id}.html"
                path = os.path.join(html_dir, file_name)
                folder = "HTML"
            elif tenk_url.lower().endswith(".txt"):
                file_name = f"{firm_id}.txt"
                path = os.path.join(txt_dir, file_name)
                folder = "TXT"
            else:
                file_name = f"{firm_id}.dat"
                path = os.path.join(other_dir, file_name)
                folder = "Neither"

            with open(path, "wb") as f:
                f.write(tenk_resp.content)

            saved.append(path)
            print(f"[OK] {i+1}/{len(urls)} {firm_id} -> {path} ({folder})")

        except Exception as e:
            msg = str(e)
            errors.append({"firm_ID_Annual": firm_id, "source_url": raw, "error": msg})
            print(f"[ERR] {i+1}/{len(urls)} {firm_id}: {msg}")

    end_t = time.time()
    print(f"\n耗時 {end_t - start_t:.2f} 秒；成功 {len(saved)} 檔，失敗 {len(errors)} 檔")

    if errors:
        print("\n失敗詳情：")
        for e in errors:
            print(f"firm_ID_Annual={e['firm_ID_Annual']}, URL={e['source_url']}, Error={e['error']}")

    return saved


df_slice = res.iloc[1:2]
saved_paths = download_10k_from_df(df_slice, limit=len(df_slice))
print(len(saved_paths), saved_paths[:3])


NameError: name 'res' is not defined

# 把資料夾的10萬份資料分成10份

In [8]:
import os
import shutil
from pathlib import Path
from tqdm import tqdm  # pip install tqdm
from datetime import datetime

# === 參數 ===
src_dir = Path("/Volumes/One Touch/10_k_file_HTML")
dst_base = Path("/Volumes/One Touch/10_k_file_split")
chunk_size = 10000
error_log = Path("move_errors.txt")  # 錯誤檔案清單輸出路徑
only_html = False  # 若要只搬 .html，改成 True

# === 準備 ===
dst_base.mkdir(parents=True, exist_ok=True)
if error_log.exists():
    # 避免覆蓋舊紀錄，改名備份
    error_log.rename(error_log.with_suffix(".bak"))

def want(p: Path) -> bool:
    if not p.is_file():
        return False
    if p.name.startswith("._"):  # AppleDouble
        return False
    if p.name == ".DS_Store":
        return False
    if only_html and p.suffix.lower() != ".html":
        return False
    return True

all_files = [p.name for p in src_dir.iterdir() if want(p)]
all_files.sort()
n_files = len(all_files)
n_parts = (n_files + chunk_size - 1) // chunk_size

print(f"總共 {n_files} 個檔案；每份 {chunk_size}；將切成 {n_parts} 份")
print(f"錯誤將記錄到：{error_log.resolve()}")

errs = 0
skipped = 0
moved = 0

with tqdm(total=n_files, desc="moving", unit="file") as bar, open(error_log, "a", encoding="utf-8") as elog:
    elog.write(f"# Move errors @ {datetime.now().isoformat()}\n")
    elog.write(f"# src_dir={src_dir}\n# dst_base={dst_base}\n# chunk_size={chunk_size}\n\n")

    for i in range(n_parts):
        part_dir = dst_base / f"part_{i+1}"
        part_dir.mkdir(parents=True, exist_ok=True)

        start = i * chunk_size
        end = min((i+1) * chunk_size, n_files)

        for fname in all_files[start:end]:
            src = src_dir / fname
            dst = part_dir / fname
            try:
                if dst.exists():
                    skipped += 1
                    bar.update(1)
                    continue
                shutil.move(str(src), str(dst))
                moved += 1
            except FileNotFoundError as e:
                errs += 1
                elog.write(f"FileNotFoundError\t{src} -> {dst}\t{e}\n")
            except Exception as e:
                errs += 1
                elog.write(f"Exception\t{src} -> {dst}\t{repr(e)}\n")
            finally:
                bar.update(1)

print("\n分割完成 ✅")
print(f"成功搬移: {moved}，已存在略過: {skipped}，錯誤: {errs}")
print(f"詳細錯誤清單：{error_log.resolve()}")


總共 108003 個檔案；每份 10000；將切成 11 份
錯誤將記錄到：/Users/wanghao/10k/move_errors.txt


moving: 100%|████████████████████████| 108003/108003 [10:45<00:00, 167.44file/s]


分割完成 ✅
成功搬移: 108003，已存在略過: 0，錯誤: 0
詳細錯誤清單：/Users/wanghao/10k/move_errors.txt


# 把"cik"和"accession" 結合組成URL

# 透過這個URL去下載8k

# 用10000筆去分87萬筆資料